# IDEA #8: Review -> Return Correlation

Muc tieu: Kiem chung gia thuyet don co rating thap co xac suat return cao hon, theo khung D-Di-P-Pr.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from scipy.stats import fisher_exact

sns.set_theme(style='whitegrid')
np.random.seed(42)

cwd = Path.cwd()
DATA_DIR = None
for _ in range(6):
    candidate = cwd / 'data' / 'datathon-2026-round-1'
    if candidate.exists():
        DATA_DIR = candidate
        break
    cwd = cwd.parent

if DATA_DIR is None:
    raise FileNotFoundError('Khong tim thay data/datathon-2026-round-1')

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Data path: {DATA_DIR}')
print(f'Output path: {OUTPUT_DIR.resolve()}')

## CELL 2: Load du lieu

In [ ]:
orders = pd.read_csv(DATA_DIR / 'orders.csv')
reviews = pd.read_csv(DATA_DIR / 'reviews.csv')
returns = pd.read_csv(DATA_DIR / 'returns.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')

orders['order_date'] = pd.to_datetime(orders['order_date'])
reviews['review_date'] = pd.to_datetime(reviews['review_date'])
returns['return_date'] = pd.to_datetime(returns['return_date'])

print('Orders shape:', orders.shape)
print('Reviews shape:', reviews.shape)
print('Returns shape:', returns.shape)
print('Products shape:', products.shape)

## CELL 3: Tao fact table review-return

In [ ]:
review_order = reviews.groupby('order_id').agg(
    avg_rating=('rating', 'mean'),
    min_rating=('rating', 'min'),
    review_count=('review_id', 'count'),
    first_review_date=('review_date', 'min')
).reset_index()

returns_order = returns.groupby('order_id').agg(
    returned_flag=('return_id', 'count'),
    total_refund=('refund_amount', 'sum'),
    first_return_date=('return_date', 'min')
).reset_index()
returns_order['returned_flag'] = (returns_order['returned_flag'] > 0).astype(int)

orders_review = orders.merge(review_order, on='order_id', how='left')
orders_review = orders_review.merge(returns_order[['order_id', 'returned_flag', 'total_refund', 'first_return_date']], on='order_id', how='left')
orders_review['returned_flag'] = orders_review['returned_flag'].fillna(0).astype(int)
orders_review['total_refund'] = orders_review['total_refund'].fillna(0)

# Chi giu nhom co review de phan tich tương quan review-return
reviewed = orders_review[orders_review['avg_rating'].notna()].copy()

def rating_band(x):
    if x <= 2:
        return 'Low (<=2)'
    if x < 4:
        return 'Mid (2-<4)'
    return 'High (>=4)'

reviewed['rating_band'] = reviewed['avg_rating'].apply(rating_band)

band_summary = reviewed.groupby('rating_band').agg(
    orders_count=('order_id', 'nunique'),
    returned_orders=('returned_flag', 'sum'),
    avg_refund=('total_refund', 'mean')
).reset_index()
band_summary['return_rate'] = band_summary['returned_orders'] / band_summary['orders_count']

# Category-level: dua return theo order_id vao category tu review product
review_cat = reviews.merge(products[['product_id', 'category']], on='product_id', how='left')
review_cat = review_cat.merge(returns_order[['order_id', 'returned_flag']], on='order_id', how='left')
review_cat['returned_flag'] = review_cat['returned_flag'].fillna(0).astype(int)
review_cat['rating_band'] = review_cat['rating'].apply(lambda x: 'Low (<=2)' if x <= 2 else ('Mid (2-<4)' if x < 4 else 'High (>=4)'))

cat_band = review_cat.groupby(['category', 'rating_band']).agg(
    review_lines=('review_id', 'count'),
    returned_lines=('returned_flag', 'sum')
).reset_index()
cat_band['return_rate'] = np.where(cat_band['review_lines'] > 0, cat_band['returned_lines'] / cat_band['review_lines'], 0)

print('Reviewed orders:', reviewed.shape[0])
print('Band summary rows:', band_summary.shape[0])
print('Category-band rows:', cat_band.shape[0])

## CELL 4: TANG 1 - DESCRIPTIVE

In [ ]:
print('='*72)
print('TANG 1: DESCRIPTIVE - REVIEW & RETURN OVERVIEW')
print('='*72)

total_orders = orders['order_id'].nunique()
reviewed_orders = reviewed['order_id'].nunique()
review_coverage = reviewed_orders / total_orders if total_orders > 0 else 0
overall_reviewed_return_rate = reviewed['returned_flag'].mean()

print(f'Tong orders: {total_orders:,.0f}')
print(f'Orders co review: {reviewed_orders:,.0f} ({review_coverage:.2%})')
print(f'Return rate trong nhom co review: {overall_reviewed_return_rate:.2%}')

print('\nReturn rate theo rating band:')
for _, r in band_summary.sort_values('return_rate', ascending=False).iterrows():
    print(f"- {r['rating_band']:10} | return_rate={r['return_rate']:.2%} | orders={r['orders_count']:>7,.0f}")

## CELL 5: TANG 2 - DIAGNOSTIC

In [ ]:
print('='*72)
print('TANG 2: DIAGNOSTIC - LOW RATING CO THAT SU RUI RO HON?')
print('='*72)

low_group = reviewed[reviewed['avg_rating'] <= 2]
high_group = reviewed[reviewed['avg_rating'] >= 4]

low_return = int(low_group['returned_flag'].sum())
low_not_return = int(low_group.shape[0] - low_return)
high_return = int(high_group['returned_flag'].sum())
high_not_return = int(high_group.shape[0] - high_return)

contingency = np.array([[low_return, low_not_return], [high_return, high_not_return]])
odds_ratio, p_value = fisher_exact(contingency, alternative='greater')

low_rate = low_group['returned_flag'].mean() if len(low_group) > 0 else np.nan
high_rate = high_group['returned_flag'].mean() if len(high_group) > 0 else np.nan
rate_ratio = (low_rate / high_rate) if (pd.notna(low_rate) and pd.notna(high_rate) and high_rate > 0) else np.nan

print(f'Low rating (<=2) return rate: {low_rate:.2%}')
print(f'High rating (>=4) return rate: {high_rate:.2%}')
print(f'Rate ratio (low/high): {rate_ratio:.2f}x')
print(f'Fisher exact p-value: {p_value:.6f}')
print(f'Odds ratio: {odds_ratio:.2f}')

top_risk_cat = cat_band[cat_band['rating_band'] == 'Low (<=2)'].sort_values('return_rate', ascending=False).head(3)
print('\nTop 3 category rui ro cao trong nhom low rating:')
for _, r in top_risk_cat.iterrows():
    print(f"- {r['category']:12} | return_rate={r['return_rate']:.2%} | review_lines={r['review_lines']}")

## CELL 6: TANG 3 - PREDICTIVE

In [ ]:
reviewed['month'] = reviewed['order_date'].dt.to_period('M').dt.to_timestamp()
monthly_low = reviewed[reviewed['avg_rating'] <= 2].groupby('month').agg(
    orders_count=('order_id', 'nunique'),
    returned_orders=('returned_flag', 'sum')
).reset_index()
monthly_low['return_rate'] = np.where(monthly_low['orders_count'] > 0, monthly_low['returned_orders'] / monthly_low['orders_count'], 0)

X = np.arange(len(monthly_low)).reshape(-1, 1)
y = monthly_low['return_rate'].values
model = LinearRegression()
model.fit(X, y)
future_X = np.arange(len(monthly_low), len(monthly_low) + 3).reshape(-1, 1)
forecast_return = model.predict(future_X)
trend_slope = model.coef_[0]

print('='*72)
print('TANG 3: PREDICTIVE - FORECAST RETURN RATE NHOM LOW RATING')
print('='*72)
print(f'Current low-rating return rate: {monthly_low.iloc[-1]["return_rate"]:.2%}')
print(f'Forecast +1M: {forecast_return[0]:.2%}')
print(f'Forecast +2M: {forecast_return[1]:.2%}')
print(f'Forecast +3M: {forecast_return[2]:.2%}')
print(f'Trend slope: {trend_slope*100:.4f} pp/thang')

## CELL 7: CHART 1 - Return rate theo rating band

In [ ]:
plot1 = band_summary.sort_values('return_rate', ascending=False).copy()
plt.figure(figsize=(9, 5))
sns.barplot(data=plot1, x='rating_band', y='return_rate', hue='rating_band', dodge=False, palette='rocket', legend=False)
plt.title('Return rate theo rating band')
plt.xlabel('Rating band')
plt.ylabel('Return rate')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_return_rate_by_rating_band.png', dpi=200)
plt.show()
print('Saved: 01_return_rate_by_rating_band.png')

## CELL 8: CHART 2 - Heatmap category x rating band

In [ ]:
heat = cat_band.pivot(index='category', columns='rating_band', values='return_rate').fillna(0)
plt.figure(figsize=(8, 5))
sns.heatmap(heat, annot=True, fmt='.2%', cmap='YlOrRd')
plt.title('Return rate theo category x rating band')
plt.xlabel('Rating band')
plt.ylabel('Category')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_category_rating_return_heatmap.png', dpi=200)
plt.show()
print('Saved: 02_category_rating_return_heatmap.png')

## CELL 9: CHART 3 - Monthly return trend low vs high rating

In [ ]:
monthly_band = reviewed.copy()
monthly_band['band_2'] = np.where(monthly_band['avg_rating'] <= 2, 'Low (<=2)', np.where(monthly_band['avg_rating'] >= 4, 'High (>=4)', 'Mid'))
monthly_band = monthly_band[monthly_band['band_2'].isin(['Low (<=2)', 'High (>=4)'])]
monthly_trend = monthly_band.groupby(['month', 'band_2']).agg(
    orders_count=('order_id', 'nunique'),
    returned_orders=('returned_flag', 'sum')
).reset_index()
monthly_trend['return_rate'] = np.where(monthly_trend['orders_count'] > 0, monthly_trend['returned_orders'] / monthly_trend['orders_count'], 0)

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_trend, x='month', y='return_rate', hue='band_2')
plt.title('Xu huong return rate theo thang: Low rating vs High rating')
plt.xlabel('Thang')
plt.ylabel('Return rate')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_low_high_rating_return_trend.png', dpi=200)
plt.show()
print('Saved: 03_low_high_rating_return_trend.png')

## CELL 10: CHART 4 - Low rating return trend + forecast

In [ ]:
hist = monthly_low.copy()
future_months = pd.date_range(hist['month'].max() + pd.offsets.MonthBegin(1), periods=3, freq='MS')

plt.figure(figsize=(12, 6))
plt.plot(hist['month'], hist['return_rate'], label='Thuc te', color='#1d3557')
plt.plot(future_months, forecast_return, marker='o', linestyle='--', label='Forecast (+3M)', color='#e63946')
plt.title('Low-rating return rate: trend va du bao 3 thang')
plt.xlabel('Thang')
plt.ylabel('Return rate')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_low_rating_return_forecast.png', dpi=200)
plt.show()
print('Saved: 04_low_rating_return_forecast.png')

## CELL 11: TANG 4 - PRESCRIPTIVE

In [ ]:
print('='*72)
print('TANG 4: PRESCRIPTIVE - CSKH TRIGGER SAU REVIEW THAP')
print('='*72)

low_returns = int(low_return)
avg_refund_per_return = returns['refund_amount'].mean() if len(returns) > 0 else 0
potential_saved_returns = low_returns * 0.20
potential_refund_saved = potential_saved_returns * avg_refund_per_return

print('1) Trigger CSKH <=48h sau review <=2 sao')
print(f'- Return rate low rating hien tai: {low_rate:.2%}')
print(f'- Muc tieu giam return 20% trong nhom low rating')

print('2) Priority queue theo category co risk cao o nhom low rating')
for _, r in top_risk_cat.iterrows():
    print(f"- {r['category']}: {r['return_rate']:.2%}")

print('3) Uoc tinh tac dong neu giam 20% return nhom low rating:')
print(f'- So return co the tranh duoc: {potential_saved_returns:,.0f}')
print(f'- So tien refund co the tiet kiem: ${potential_refund_saved:,.0f}')

print('4) Van hanh scorecard: low-rating volume, response SLA 48h, return_after_review, refund_saved')

## CELL 12: Export summary metrics

In [ ]:
summary_metrics = pd.DataFrame({
    'Metric': [
        'Total Orders',
        'Reviewed Orders',
        'Review Coverage',
        'Overall Reviewed Return Rate',
        'Low Rating Return Rate',
        'High Rating Return Rate',
        'Rate Ratio Low/High',
        'Fisher Exact p-value',
        'Odds Ratio',
        'Top Risk Category (Low Rating)',
        'Top Risk Category Return Rate',
        'Low Rating Forecast (+1M)',
        'Low Rating Trend Slope (pp/month)',
        'Low Group Return Volume',
        'Estimated Saved Returns (20%)',
        'Avg Refund Per Return',
        'Estimated Refund Saved',
        'CSKH Trigger SLA',
        'Primary Action',
        'Business One-liner'
    ],
    'Value': [
        f"{total_orders:,.0f}",
        f"{reviewed_orders:,.0f}",
        f"{review_coverage:.2%}",
        f"{overall_reviewed_return_rate:.2%}",
        f"{low_rate:.2%}",
        f"{high_rate:.2%}",
        f"{rate_ratio:.2f}x",
        f"{p_value:.6f}",
        f"{odds_ratio:.2f}",
        str(top_risk_cat.iloc[0]['category']) if len(top_risk_cat) > 0 else 'N/A',
        f"{top_risk_cat.iloc[0]['return_rate']:.2%}" if len(top_risk_cat) > 0 else 'N/A',
        f"{forecast_return[0]:.2%}",
        f"{trend_slope*100:.4f}",
        f"{low_returns:,.0f}",
        f"{potential_saved_returns:,.0f}",
        f"${avg_refund_per_return:,.2f}",
        f"${potential_refund_saved:,.0f}",
        '<=48h',
        'CSKH post-review low rating',
        'Review thap la SOS, xu ly som giam return va bao ve danh tieng'
    ]
})

summary_metrics.to_csv(OUTPUT_DIR / 'summary_metrics.csv', index=False)
print(summary_metrics.to_string(index=False))
print('\nSaved: summary_metrics.csv')

## CELL 13: Ket luan

In [ ]:
print('='*72)
print('ANALYSIS COMPLETE - IDEA #8 REVIEW RETURN CORRELATION')
print('='*72)
print('One-liner: Review thap la SOS, khong chi la feedback; xu ly som co the giam return dang ke.')
print('Da tao 4 bieu do + summary_metrics.csv trong thu muc outputs.')